In [1]:
# Vector Data impimentation
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")

C:\Users\CHAMA COMPUTERS\AppData\Local\Temp\ipykernel_44612\53335358.py:6: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [2]:
loader = PyPDFLoader("../test_documents/attention.pdf")

In [3]:
document = loader.load()

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200)
documents =  text_splitter.split_documents(document)
vectors = FAISS.from_documents(documents, embeddings)

In [5]:
retriver = vectors.as_retriever()

In [6]:
from langchain_core.tools import tool

@tool
def database_search(query: str) -> str:
    """Search for information about the research paper.
    You should use this tool to get the information about the relevant research paper."""
    docs = retriver.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

In [7]:
parse_agent_Tools = [database_search]

# Agent Implimentation 

# Document Parse Agent

In [8]:
## Agents
from langchain.agents import create_agent
from langchain.tools import tool


from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
load_dotenv()


True

In [9]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2, 
    # other params...
)
parse_llm_with_tools = llm.bind_tools(parse_agent_Tools)

In [10]:
parse_agent_prompt = """You are the Research Document Parsing Agent. 
First Use the given retriver_tool Tool to get research paper context and  You Read the given  research paper  Fully and divide them into Abstract, Method, Math, Experiments, Results.
Rules:
- Do NOT wait for user input or paper name
- Go directly to the vector database and extract
- Do NOT ask questions
- Use the document retrieval tool to read the paper
- Be concise but complete
"""

In [11]:
parse_agent = create_agent(
    model=parse_llm_with_tools,  # or any other supported model identifier string
    tools=parse_agent_Tools,
    system_prompt=parse_agent_prompt
)

In [14]:
inputs = {"messages": [{"role": "user", "content": "Analyze the given research paper"}]}
result_parse_agent=parse_agent.invoke(inputs)
print(result_parse_agent["messages"][-1].content)

**Attention Is All You Need** – Ashish Vaswani et al. (2017)

---

### 1. Abstract  
The paper introduces the **Transformer**, a sequence‑to‑sequence model that replaces recurrent and convolutional layers with a stack of multi‑head self‑attention modules. The architecture is fully parallelizable, trains faster, and achieves state‑of‑the‑art BLEU scores on WMT 2014 English‑German and English‑French translation tasks. The authors claim that the Transformer outperforms previous models while requiring fewer computational resources.

---

### 2. Method (Model Architecture)

| Component | Description |
|-----------|-------------|
| **Encoder** | 6 identical layers. Each layer has: <br>• *Multi‑head self‑attention* (queries, keys, values all from encoder output). <br>• *Position‑wise feed‑forward network* (two linear layers with ReLU). |
| **Decoder** | 6 identical layers. Each layer has: <br>• *Masked multi‑head self‑attention* (preventing future‑token leakage). <br>• *Encoder‑decoder attent

In [15]:
result_parse_agent

{'messages': [HumanMessage(content='Analyze the given research paper', additional_kwargs={}, response_metadata={}, id='48f90933-1367-4c30-84d9-7e239507b012'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the database_search tool to retrieve the research paper. But we don\'t have a query. The instruction says "Do NOT wait for user input or paper name". So we need to decide a query. Maybe we can search for a generic research paper? But we need to analyze a given research paper. The user didn\'t provide a title. The instruction says "First Use the given retriver_tool Tool to get research paper context and You Read the given research paper Fully and divide them into Abstract, Method, Math, Experiments, Results." So we need to fetch a paper. But which one? The instruction says "Do NOT wait for user input or paper name". So we must choose a paper. Maybe we can search for a recent paper on a topic. But we need to produce a structured summary. We can pick a w

# Teaching Agent

In [16]:
teaching_agent_prompt = """You are the Teaching Assistant Agent. 
Read the given context carefully and explain the key concepts, Architecture ideas , algorithm intuition using 
simple language, Analogies and step by step reasoning so even a complete beginner could understand.Use the tools only if you want.
Compress the result into bullet points.
Keep only essential facts, findings, and conclusions.
"""

In [17]:
wiki_api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)
wiki_tool = WikipediaQueryRun(api_wrapper= wiki_api_wrapper)

In [18]:
llm = ChatGroq(
    model="openai/gpt-oss-safeguard-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2, 
    # other params...
)
teach_tools = [wiki_tool]
teach_llm=llm.bind_tools(teach_tools)

In [19]:
teach_agent = create_agent(
    model=teach_llm,  # or any other supported model identifier string
    tools=teach_tools,
    system_prompt=teaching_agent_prompt
)

In [20]:
teach_agent_input = {"messages": [{"role": "user", "content": "Teach me this research paper context"+ "context - "+str(result_parse_agent["messages"][-1].content)}]}
teach_agent_results=teach_agent.invoke(teach_agent_input)
print(teach_agent_results["messages"][-1].content)

**Attention Is All You Need – 2017 (Vaswani et al.) – Quick‑Start Guide**

---

### 1. Why a new model?  
- **Old ways**: Recurrent (LSTM/GRU) or convolutional nets had to process words one after another → slow, hard to parallelize.  
- **Goal**: Build a *fully parallel* sequence‑to‑sequence model that still understands long‑range word relationships.  
- **Result**: The **Transformer** – faster to train, same or better translation quality.

---

### 2. Core Idea: *Self‑Attention*  
- **What it does**: For every word, look at every other word in the sentence and decide how much it should “pay attention” to each.  
- **Analogy**: Imagine a classroom where each student (word) can instantly see every other student’s notes and decide which notes matter most for answering a question.  
- **Benefit**: No need to wait for previous words → all words processed at once.

---

### 3. Building Blocks

| Block | What it is | Why it matters |
|-------|------------|----------------|
| **Token Embeddin

In [21]:
teach_agent_results

{'messages': [HumanMessage(content='Teach me this research paper contextcontext - **Attention Is All You Need** – Ashish\u202fVaswani\u202fet\u202fal. (2017)\n\n---\n\n### 1. Abstract  \nThe paper introduces the **Transformer**, a sequence‑to‑sequence model that replaces recurrent and convolutional layers with a stack of multi‑head self‑attention modules. The architecture is fully parallelizable, trains faster, and achieves state‑of‑the‑art BLEU scores on WMT\u202f2014 English‑German and English‑French translation tasks. The authors claim that the Transformer outperforms previous models while requiring fewer computational resources.\n\n---\n\n### 2. Method (Model Architecture)\n\n| Component | Description |\n|-----------|-------------|\n| **Encoder** | 6 identical layers. Each layer has: <br>• *Multi‑head self‑attention* (queries, keys, values all from encoder output). <br>• *Position‑wise feed‑forward network* (two linear layers with ReLU). |\n| **Decoder** | 6 identical layers. Each 

# Mathematical Reasoning Agent

In [22]:
Math_agent_prompt = """You are the Mathematical Reasoning Agent. 
Read the given research paper context carefully, extract all the mathematical formulas and explain , analyze equations, Loss functions, 
optimization methods.
Explain what each equation represents, why it needed and assumptions made so even a complete beginner could understand.
present the results in a human-readable format. 
Compress the result into bullet points.
Keep only essential facts, findings, and conclusions.
Use the provided tools ONLY if you need.

"""

In [23]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2, 
    # other params...
)
math_tools = [wiki_tool]
math_llm=llm.bind_tools(math_tools)

In [24]:
math_agent = create_agent(
    model=math_llm,  # or any other supported model identifier string
    tools=math_tools,
    system_prompt=Math_agent_prompt
)

In [25]:
math_agent_input = {"messages": [{"role": "user", "content": "explain all the mathematical terms in this research paper"+ "context - "+str(result_parse_agent["messages"][-1].content)}]}
math_agent_results=teach_agent.invoke(math_agent_input)
print(math_agent_results["messages"][-1].content)

**Key Mathematical Ideas in “Attention Is All You Need” – Simplified**

---

### 1. **Vectors & Matrices**
- **Token embedding**: each word → a *vector* of numbers (`d_model` dimensions).  
- **Matrices** (`W`, `b`): collections of numbers that transform vectors (e.g., change size, mix features).

---

### 2. **Dot‑Product & Softmax**
- **Dot‑product** (`QKᵀ`): measures similarity between two vectors.  
  *Analogy*: think of two arrows; the dot product tells how “aligned” they are.  
- **Softmax**: turns a list of numbers into probabilities that sum to 1.  
  *Analogy*: a voting system where higher scores get more votes.

---

### 3. **Scaled Dot‑Product Attention**
\[
\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
\]

- **Q (query)**, **K (key)**, **V (value)**: all are matrices derived from the same input (self‑attention) or from encoder output (encoder‑decoder attention).  
- **Scaling by \(1/\sqrt{d_k}\)**: prevents huge dot‑product values that w

In [26]:
math_agent_results

{'messages': [HumanMessage(content='explain all the mathematical terms in this research papercontext - **Attention Is All You Need** – Ashish\u202fVaswani\u202fet\u202fal. (2017)\n\n---\n\n### 1. Abstract  \nThe paper introduces the **Transformer**, a sequence‑to‑sequence model that replaces recurrent and convolutional layers with a stack of multi‑head self‑attention modules. The architecture is fully parallelizable, trains faster, and achieves state‑of‑the‑art BLEU scores on WMT\u202f2014 English‑German and English‑French translation tasks. The authors claim that the Transformer outperforms previous models while requiring fewer computational resources.\n\n---\n\n### 2. Method (Model Architecture)\n\n| Component | Description |\n|-----------|-------------|\n| **Encoder** | 6 identical layers. Each layer has: <br>• *Multi‑head self‑attention* (queries, keys, values all from encoder output). <br>• *Position‑wise feed‑forward network* (two linear layers with ReLU). |\n| **Decoder** | 6 id

# Experimental Analysis Agent

In [27]:
experimental_analysis_agent_prompt = """You are the Experimental Analysis Agent and Empirical reviewer. 
Read the given context carefully and explain / Reviews Datasets used, Experimental setup, Baselines, Evaluation metrics.
Determines whether experiments support claims, If comparisons are fair. Conpare them with other related research papers.
Compress the result into bullet points.
Keep only essential facts, findings, and conclusions.
Use the given tools only if needed. Give me exacly a final answer don't think too much.
"""

In [28]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper
arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv_tool = ArxivQueryRun(api_wrapper= arxiv_wrapper)

In [29]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2, 
    # other params...
)
experimental_analysis_agent_tools = [arxiv_tool,wiki_tool]
experimental_analysis_agent_llm=llm.bind_tools(experimental_analysis_agent_tools)

In [30]:
experimental_analysis_agent = create_agent(
    model=experimental_analysis_agent_llm,  # or any other supported model identifier string
    tools=experimental_analysis_agent_tools,
    system_prompt=experimental_analysis_agent_prompt
)

In [31]:
experimental_analysis_agent_input = {"messages": [{"role": "user", "content": "experiment analysis on the given research paper"+ "context - "+str(result_parse_agent["messages"][-1].content)}]}
experimental_analysis_agent_results=experimental_analysis_agent.invoke(experimental_analysis_agent_input)
print(experimental_analysis_agent_results["messages"][-1].content)

**Experimental Review – “Attention Is All You Need” (Vaswani et al., 2017)**  

**Datasets**  
- **WMT 2014 English‑German (EN‑DE)** – ~4.5 M sentence pairs, standard test set used in NMT benchmarks.  
- **WMT 2014 English‑French (EN‑FR)** – ~36 M sentence pairs, same benchmark split.  
- No additional low‑resource or domain‑specific corpora; focus is on large‑scale, high‑resource translation.

**Experimental Setup**  
- **Model variants**: *Transformer‑Base* (d_model = 512, h = 8, d_ff = 2048, 6 encoder + 6 decoder layers) and *Transformer‑Big* (d_model = 1024, h = 16, d_ff = 4096) (big not shown in table but reported in paper).  
- **Training**: 1 M steps, Adam optimizer (β₁ = 0.9, β₂ = 0.98, ε = 10⁻⁹), custom learning‑rate schedule `lr = d_model⁻⁰·⁵ * min(step⁻⁰·⁵, step * warmup⁻¹·⁵)` with warm‑up = 4000 steps.  
- **Batching**: 4096 tokens per GPU, 8 × NVIDIA V100 GPUs (effective batch ≈ 32 k tokens).  
- **Regularization**: Dropout = 0.1 on embeddings, attention weights, and feed‑

In [32]:
experimental_analysis_agent_results

{'messages': [HumanMessage(content='experiment analysis on the given research papercontext - **Attention Is All You Need** – Ashish\u202fVaswani\u202fet\u202fal. (2017)\n\n---\n\n### 1. Abstract  \nThe paper introduces the **Transformer**, a sequence‑to‑sequence model that replaces recurrent and convolutional layers with a stack of multi‑head self‑attention modules. The architecture is fully parallelizable, trains faster, and achieves state‑of‑the‑art BLEU scores on WMT\u202f2014 English‑German and English‑French translation tasks. The authors claim that the Transformer outperforms previous models while requiring fewer computational resources.\n\n---\n\n### 2. Method (Model Architecture)\n\n| Component | Description |\n|-----------|-------------|\n| **Encoder** | 6 identical layers. Each layer has: <br>• *Multi‑head self‑attention* (queries, keys, values all from encoder output). <br>• *Position‑wise feed‑forward network* (two linear layers with ReLU). |\n| **Decoder** | 6 identical la

# Testing Multi Agent Architecture

In [41]:
from langgraph_supervisor import create_supervisor
from langchain.chat_models import init_chat_model

In [42]:
llm_sup = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2, 
    # other params...
)

In [44]:
SUPERVISOR_PROMPT = """You are the research coordinator agent. 
    Read the given research paper context
    You can Breaks the paper into logical components such as problem, Method, Math (use the teach_math_tool for this),   
    Theory (use the teach_paper_tool for this), Experiments (use the experiment_analysis_tool for this), Results
    Assigns tasks to specialist agents, Controls analysis order, Provide agent inputs only with the relavant contenxt using the given tool.
    Give ONLY relevent CONTEXT TO EACH AGENT / TOOLS WITH INPUT.
    teach_math_tool - to explain math
    teach_paper_tool - to explain the paper content
    Decides when analysis is “complete” 
    Break down user requests into appropriate tool calls and coordinate the results. 
    When a request involves multiple actions, use multiple tools in sequence.
    finally you should give a full blown analysis descriptive report . Not Just A Summary.
    Full blown agent analysis
"""


In [45]:
supervisor = create_supervisor(
    model=llm_sup,
    agents=[teach_agent, math_agent,experimental_analysis_agent],
    prompt=(
        "You are a supervisor managing two agents:\n"
        "- a research agent. Assign research-related tasks to this agent\n"
        "- a math agent. Assign math-related tasks to this agent\n"
        "Assign work to one agent at a time, do not call agents in parallel.\n"
        "Do not do any work yourself."
    ),
    add_handoff_back_messages=True,
    output_mode="full_history",
).compile()

ValueError: Please specify a name when you create your agent, either via `create_react_agent(..., name=agent_name)` or via `graph.compile(name=name)`.

In [ ]:
from IPython.display import display, Image
display(Image(supervisor.get_graph().draw_mermaid_png()))

In [44]:
from langchain.tools import tool

In [ ]:
You are the Teaching Assistant Agent. 
Read the given context carefully and explain the key concepts, Architecture ideas , algorithm intuition using 
simple language, Analogies and step by step reasoning so even a complete beginner could understand.Use the tools only if you want.
Input: {input}
{agent_scratchpad}
<context>
{context}
</context>

In [46]:
@tool
async def teach_paper_tool(request: str, context: str) -> str:
    """Teach the given research paper context in a beginner friendly manner.

    Use this tool to explain the key concepts, Architecture ideas , algorithm intuition using 
    simple language, Analogies and step by step reasoning so even a complete beginner could understand.
    Compress the output content into bullet points.
    Keep only essential facts, findings, and conclusions.

    """
    combined_input = f"""
TASK:
{request}

RESEARCH PAPER CONTEXT:
{context}
"""
    result = await teaching_agent_executor.ainvoke({
        "input": combined_input
    })

    return result['output']




In [47]:
@tool
async def teach_math_tool(request: str, context: str) -> str:
    """Read the given research paper context carefully, extract all the mathematical formulas and explain , analyze equations, Loss functions, 
    optimization methods.

    Use this tool to Explain what each equation represents, why it needed and assumptions made so even a complete beginner could understand.
    present the results in a human-readable format.

    """
    combined_input = f"""
    TASK:
    {request}
    
    RESEARCH PAPER CONTEXT:
    {context}
    """
    result = await math_agent_executor.ainvoke({
        "input": combined_input
    })

    return result['output']

In [48]:
@tool
async def experiment_analysis_tool(request: str, context: str) -> str:
    """
    Use this tool to conduct the Experimental Analysis
    Read the given context carefully and explain / Reviews Datasets used, Experimental setup, Baselines, Evaluation metrics.
    Determines whether experiments support claims, If comparisons are fair. Conpare them with other related research papers.
    Only use a sufficient token amount so supervisor agent can handle it
    

    """
    combined_input = f"""
    TASK:
    {request}
    
    RESEARCH PAPER CONTEXT:
    {context}
    """
    result = await experimental_analysis_agent_executor.ainvoke({
        "input": combined_input
    })

    return result['output']

# Summarizer Agent

In [49]:
summarizer_agent_prompt = ChatPromptTemplate.from_template("""
    Compress the following content into bullet points.
    Keep only essential facts, findings, and conclusions.
    
    Content:
    {input}
    {agent_scratchpad}
    """) 

summary_llm = ChatGroq(
    model="openai/gpt-oss-safeguard-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params... 
)


#supervisor_tools = [teach_paper_tool, teach_math_tool, experiment_analysis_tool]
#supervisor_llm_with_tools =llm_supervisor.bind_tools(supervisor_tools)
summary_agent = create_tool_calling_agent(summary_llm,[wiki_tool], summarizer_agent_prompt)
summary_agent_executor = AgentExecutor(agent=summary_agent, tools=[wiki_tool], verbose=True,max_iterations=2,handle_parsing_errors=True)

C:\Anaconda3\Lib\site-packages\langchain_groq\chat_models.py:359: UserWarning: WARNING! reasoning_format is not default parameter.
                    reasoning_format was transferred to model_kwargs.
                    Please confirm that reasoning_format is what you intended.
  warnings.warn(


In [50]:
summarized_context = summary_agent_executor.invoke({"input":result_doc_agent["output"]})



> Entering new AgentExecutor chain...
**Attention Is All You Need – Key Points**

- **Core Idea**  
  - Introduces the *Transformer*, a seq‑to‑seq model that uses only attention mechanisms (no recurrence or convolution).  
  - Consists of an encoder and decoder, each with 6 identical layers of multi‑head self‑attention + position‑wise feed‑forward networks, plus residual + layer‑norm.

- **Architecture Highlights**  
  - **Encoder**: multi‑head self‑attention → feed‑forward.  
  - **Decoder**: masked self‑attention → encoder‑decoder attention → feed‑forward.  
  - **Multi‑head Attention**: split into 8 heads, scaled dot‑product, concatenate, linear projection.  
  - **Positional Encoding**: sinusoidal vectors added to embeddings (learned variants work similarly).  
  - **Output**: linear layer + softmax over vocabulary.

- **Mathematical Core**  
  - Scaled dot‑product: `Attention(Q,K,V)=softmax(QKᵀ/√dk)V`.  
  - Multi‑head: `Concat(head_i)WO` with `head_i=Attention(QWi^Q, Ki^K, Vi^V

In [51]:
print(summarized_context['output'])

**Attention Is All You Need – Key Points**

- **Core Idea**  
  - Introduces the *Transformer*, a seq‑to‑seq model that uses only attention mechanisms (no recurrence or convolution).  
  - Consists of an encoder and decoder, each with 6 identical layers of multi‑head self‑attention + position‑wise feed‑forward networks, plus residual + layer‑norm.

- **Architecture Highlights**  
  - **Encoder**: multi‑head self‑attention → feed‑forward.  
  - **Decoder**: masked self‑attention → encoder‑decoder attention → feed‑forward.  
  - **Multi‑head Attention**: split into 8 heads, scaled dot‑product, concatenate, linear projection.  
  - **Positional Encoding**: sinusoidal vectors added to embeddings (learned variants work similarly).  
  - **Output**: linear layer + softmax over vocabulary.

- **Mathematical Core**  
  - Scaled dot‑product: `Attention(Q,K,V)=softmax(QKᵀ/√dk)V`.  
  - Multi‑head: `Concat(head_i)WO` with `head_i=Attention(QWi^Q, Ki^K, Vi^V)`.  
  - Feed‑forward: `FFN(x)=max(0,xW

In [52]:
SUPERVISOR_PROMPT = ChatPromptTemplate.from_template(
    """You are the research coordinator agent. 
    Read the given research paper context
    You can Breaks the paper into logical components such as problem, Method, Math (use the teach_math_tool for this),   
    Theory (use the teach_paper_tool for this), Experiments (use the experiment_analysis_tool for this), Results
    Assigns tasks to specialist agents, Controls analysis order, Provide agent inputs only with the relavant contenxt using the given tool.
    Give ONLY relevent CONTEXT TO EACH AGENT / TOOLS WITH INPUT.
    teach_math_tool - to explain math
    teach_paper_tool - to explain the paper content
    Decides when analysis is “complete” 
    Break down user requests into appropriate tool calls and coordinate the results. 
    When a request involves multiple actions, use multiple tools in sequence.
    finally you should give a full blown analysis descriptive report . Not Just A Summary.
    Full blown agent analysis

    Input: {input}
    {agent_scratchpad}
    <context>
    {context}
    </context>"""
)

In [53]:

llm_supervisor = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)
supervisor_tools = [teach_paper_tool, teach_math_tool, experiment_analysis_tool]
supervisor_llm_with_tools =llm_supervisor.bind_tools(supervisor_tools)

C:\Anaconda3\Lib\site-packages\langchain_groq\chat_models.py:359: UserWarning: WARNING! reasoning_format is not default parameter.
                    reasoning_format was transferred to model_kwargs.
                    Please confirm that reasoning_format is what you intended.
  warnings.warn(


In [54]:
supervisor_agent = create_tool_calling_agent(supervisor_llm_with_tools,supervisor_tools, SUPERVISOR_PROMPT)
supervisor_agent_executor = AgentExecutor(agent=supervisor_agent, tools=supervisor_tools, verbose=True,max_iterations=10,handle_parsing_errors=True)

In [57]:
import asyncio

async def run_pipeline():

    result_doc_agent= await parse_agent_executor.ainvoke({'input':"analyze the  given research paper with the given database retriver tool"})


    summary_context= await summary_agent_executor.ainvoke({"input":result_doc_agent["output"]})
    
    result = await supervisor_agent_executor.ainvoke({
        "input": f"""
Analyze and explain the following research paper.
Use provided tools when appropriate.

"""
    ,'context':summary_context['output']})

    return result['output']


# Run
final_output = await run_pipeline()
print(final_output)




> Entering new AgentExecutor chain...

Invoking: `Database_Search` with `{'query': 'research paper'}`


around each of the sub-layers, followed by layer normalization. We also modify the self-attention
sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This
masking, combined with fact that the output embeddings are offset by one position, ensures that the
predictions for position ican depend only on the known outputs at positions less than i.
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
of the values, where the weight assigned to each value is computed by a compatibility function of the
query with the corresponding key.
3.2.1 Scaled Dot-Product Attention
We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of
queries and keys of dimens

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01k5t1nbyrf1baqpqqj4ncp2sq` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8414, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [56]:
print(final_output)

**Key Take‑aways from the “Attention Is All You Need” fragment**

- **Model architecture**
  - Purely attention‑based encoder–decoder stack.
  - Each layer has two sub‑layers: (1) self‑attention (masked in decoder) and (2) position‑wise feed‑forward network.
  - Residual connections + layer‑norm around every sub‑layer.

- **Scaled dot‑product attention**
  - Formula: `Attention(Q,K,V) = softmax((QKᵀ)/√d_k) V`.
  - Scaling by `1/√d_k` keeps dot‑product variance ≈ 1, preventing vanishing gradients in the softmax.

- **Multi‑head attention**
  - Parallel heads with independent linear projections `W^Q_i, W^K_i, W^V_i`.
  - Each head outputs a `d_v`‑dimensional vector; concatenated and linearly projected back to model dimension.
  - Enables simultaneous focus on different representation sub‑spaces.

- **Decoder masking**
  - Causal mask blocks a token from attending to future positions, ensuring autoregressive generation.
  - Output embeddings are offset by one position to enforce this cons